## Colab Setup

In [ ]:
!pip install tensorflow==2.16.2 numpy==1.26.4 torch==2.2.2 transformers==4.51.3 torchao optuna

In [ ]:
# 1. Restart runtime first (Runtime > Restart Runtime)
# 2. Then run this:

!pip uninstall -y torch torchvision torchaudio torchao
!pip uninstall -y transformers accelerate

# 3. Clean reinstall with matching versions
!pip install torch==2.2.2+cu121 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.51.3 accelerate

In [ ]:
from google.colab import drive
import os

mountpoint = '/content/drive/'
drive.mount(mountpoint)

import sys
sys.path.append('/content/drive/MyDrive/UoA/TPD/code/')
import gsm8k_dataset

import torch

# Load your data

train_set = torch.load('drive/MyDrive/UoA/TPD/processed_data/gsm8k_train_data.pth')
test_set = torch.load('drive/MyDrive/UoA/TPD/processed_data/gsm8k_test_data.pth')

import json
# load the train_set_labels from a file
with open('./drive/MyDrive/UoA/TPD/processed_data/train_set_labels.json', 'r') as f:
    train_labels = json.load(f)
with open('./drive/MyDrive/UoA/TPD/processed_data/test_set_labels.json', 'r') as f:
    test_labels = json.load(f)


## Generate Math Question (Before fine-tuning)

In [ ]:
# Import the class
from math_question_generator import MathQuestionGenerator

In [ ]:
# Create an instance
generator = MathQuestionGenerator()

# Generate questions
algebra_question = generator.generate_math_question(topic="algebra")
print(algebra_question["question"])
print(algebra_question["answer"])

In [ ]:
# Create an instance
generator = MathQuestionGenerator("gpt2")

# Generate questions
algebra_question = generator.generate_math_question(topic="algebra")
print(algebra_question["question"])
print(algebra_question["answer"])

In [ ]:
generative_content = generator.generate_math_question(topic="algebra")
print(generative_content)

In [ ]:
# Create an instance
generator = MathQuestionGenerator("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Generate questions
algebra_question = generator.generate_math_question(topic="algebra")
print(algebra_question["question"])
print(algebra_question["answer"])

## Importation

In [ ]:
del generate_questions_for_specific_years

In [ ]:
# reload the math_question_generator
import math_question_generator
from importlib import reload
reload(math_question_generator)

In [ ]:
import torch
from gsm8k_dataset import GSMDataset, get_examples, split_solution_into_subsentences, classify_gsm8k_solution_segments, generate_prompt
import json
from tqdm import tqdm
from datasets import Dataset
from as_es_learning import as_es_classification
from trainer import LanguageModelTrainer
from utils import get_device, extract_equations_from_text
from math_question_generator import prepare_prompt, prepare_generator, generate, generate_multiple, generate_random_student_levels, generate_questions_for_specific_years
from transformers import pipeline, GenerationConfig
from pathlib import Path
from optimizer import HyperparameterOptimizer
# from visual_optimizer import VisualHyperparameterOptimizer

## Load data (local)

In [ ]:
# read local data
train_set = torch.load('gsm8k_train_data.pth')
test_set = torch.load('gsm8k_test_data.pth')
len(train_set), len(test_set)


# load the train_set_labels from a file
with open('train_set_labels.json', 'r') as f:
    train_labels = json.load(f)
with open('test_set_labels.json', 'r') as f:
    test_labels = json.load(f)

In [ ]:
# prepare training input and output
train_prompts = []

for i in tqdm(range(len(train_set))):
    temp_result = {}
    question, answer, grade = train_set.__getitemInTxt__(i)
    transcript = train_set.__getTranscript__(i)
    types = train_set.__getTypes__(i)
    labels = train_labels[i]
    input_txt, output_txt = generate_prompt(question=question, answer=answer, grade=grade, labels=labels, transcript=transcript)
    temp_result["input"] = input_txt
    temp_result["output"] = output_txt
    train_prompts.append(temp_result)

In [ ]:
test_prompts = []

for i in tqdm(range(len(test_set))):
    temp_result = {}
    question, answer, grade = test_set.__getitemInTxt__(i)
    transcript = train_set.__getTranscript__(i)
    types = train_set.__getTypes__(i)
    labels = test_set[i]
    input_txt, output_txt = generate_prompt(question=question, answer=answer, grade=grade, labels=labels, transcript=transcript)
    temp_result["input"] = input_txt
    temp_result["output"] = output_txt
    test_prompts.append(temp_result)

In [ ]:
def split_dataset_for_exp(dataset):
    # Assume dataset is a list of dicts
    dataset = Dataset.from_list(dataset)
    print(f"Total size: {len(dataset)}")

    # Set split sizes
    train_ratio = 0.7
    val_ratio = 0.2
    # test_ratio = 0.1

    # Split test dataset for experiment
    train_end = int(train_ratio * len(dataset))
    val_end = train_end + int(val_ratio * len(dataset))


    train_data = dataset.select(range(train_end))
    val_data = dataset.select(range(train_end, val_end))
    test_data = dataset.select(range(val_end, len(dataset)))

    print(f"Train size: {len(train_data)}")
    print(f"Validation size: {len(val_data)}")
    print(f"Test size: {len(test_data)}")

    return train_data, val_data, test_data

In [ ]:
exp_train_data, exp_val_data, exp_test_data = split_dataset_for_exp(test_prompts)

## AS-ES Labelling

In [ ]:
as_es_classification = as_es_classification()

In [ ]:
test_set.__getitemInTxt__(1)

In [ ]:
train_labels, train_metrics, train_normalize_metrics = as_es_classification.label_generator(train_set)

In [ ]:
test_labels, test_metrics, test_normalize_metrics = as_es_classification.label_generator(test_set)

In [ ]:
train_labels_copy = train_labels.copy()
test_labels_copy = test_labels.copy()

In [ ]:
## save the labels
# # export train_set_labels to a file

# with open('train_set_labels.json', 'w') as f:
#     json.dump(train_labels, f, indent=4)

# with open('test_set_labels.json', 'w') as f:
#     json.dump(test_labels, f, indent=4)

# # export train_metrics and train_normalize_metrics
# with open('train_metrics.json', 'w') as f:
#     json.dump(train_metrics, f, indent=4)
# with open('train_normalize_metrics.json', 'w') as f:
#     json.dump(train_normalize_metrics, f, indent=4)
# with open('test_metrics.json', 'w') as f:
#     json.dump(test_metrics, f, indent=4)
# with open('train_normalize_metrics.json', 'w') as f:
#     json.dump(test_normalize_metrics, f, indent=4)

In [ ]:
# import json
# # load the train_set_labels from a file
# with open('train_set_labels.json', 'r') as f:
#     train_labels = json.load(f)
# with open('test_set_labels.json', 'r') as f:
#     test_labels = json.load(f)

## Experiment1 (Model selection)

In [ ]:
from huggingface_hub import login

import os
login(token=os.environ.get('HF_TOKEN'))  # Set HF_TOKEN in your environment or notebook secrets.


### finding the max_length

In [ ]:
train_token_lengths = []
for ex in tqdm(train_data, desc="Measuring token lengths"):
    text = f"<s>{ex['input']}\n{ex['output']}</s>"
    tokens = llama_tokenizer(text)["input_ids"]
    train_token_lengths.append(len(tokens))

In [ ]:
test_token_lengths = []
for ex in tqdm(test_data, desc="Measuring token lengths"):
    text = f"<s>{ex['input']}\n{ex['output']}</s>"
    tokens = llama_tokenizer(text)["input_ids"]
    test_token_lengths.append(len(tokens))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Basic statistics
print("Max tokens:", np.max(train_token_lengths))
print("Average tokens:", np.mean(train_token_lengths))
print("90th percentile:", np.percentile(train_token_lengths, 90))
print("95th percentile:", np.percentile(train_token_lengths, 95))

# Optional: Histogram
plt.hist(train_token_lengths, bins=20, color="skyblue")
plt.axvline(np.percentile(train_token_lengths, 95), color="red", linestyle="--", label="95th percentile")
plt.xlabel("Token length")
plt.ylabel("Number of examples")
plt.title("Token length distribution")
plt.legend()
plt.show()

In [ ]:
# Basic statistics
print("Max tokens:", np.max(test_token_lengths))
print("Average tokens:", np.mean(test_token_lengths))
print("90th percentile:", np.percentile(test_token_lengths, 90))
print("95th percentile:", np.percentile(test_token_lengths, 95))

# Optional: Histogram
plt.hist(test_token_lengths, bins=20, color="skyblue")
plt.axvline(np.percentile(test_token_lengths, 95), color="red", linestyle="--", label="95th percentile")
plt.xlabel("Token length")
plt.ylabel("Number of examples")
plt.title("Token length distribution")
plt.legend()
plt.show()

____

In [ ]:
from datasets import Dataset

# Format data
# Tokenization function
def tokenize_data(example, max_length=512):
    encoding = llama_tokenizer(
        "<s>" + example["input"] + "\n" + example["output"] + "</s>",
        padding=False,
        truncation=False,
        return_tensors=None,
        add_special_tokens=True
    )
    encoding["labels"] = encoding["input_ids"].copy()
    return encoding


# Tokenize all samples
tokenized_train_data = [tokenize_data(example) for example in tqdm(train_data, desc="tokenizing train data")]
tokenized_test_data = [tokenize_data(example) for example in tqdm(test_data, desc="tokenizing test data")]

tokenized_train_dataset = Dataset.from_list(tokenized_train_data)
tokenized_test_dataset = Dataset.from_list(tokenized_test_data)

In [ ]:
from datasets import Dataset


def split_dataset_for_exp(dataset):
    # Assume dataset is a list of dicts
    dataset = Dataset.from_list(dataset)
    print(f"Total size: {len(dataset)}")

    # Set split sizes
    train_ratio = 0.7
    val_ratio = 0.2
    # test_ratio = 0.1

    # Split test dataset for experiment
    train_end = int(train_ratio * len(dataset))
    val_end = train_end + int(val_ratio * len(dataset))


    train_data = dataset.select(range(train_end))
    val_data = dataset.select(range(train_end, val_end))
    test_data = dataset.select(range(val_end, len(dataset)))

    print(f"Train size: {len(train_data)}")
    print(f"Validation size: {len(val_data)}")
    print(f"Test size: {len(test_data)}")

    return train_data, val_data, test_data

In [ ]:
exp_train_data, exp_val_data, exp_test_data = split_dataset_for_exp(test_data)

In [ ]:
sample = exp_train_data[0]
print(sample.keys())  # Should include 'input_ids' and 'attention_mask'
print(type(sample['input_ids']))  # Should be list, not tensor

### LLama

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
    "target_modules": ["q_proj", "v_proj"],
}


model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llama_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

llama_trainer.setup()
final_exp_train_data = llama_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = llama_trainer.tokenize_dataset(exp_test_data)

# Train
llama_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir="./experiment/llama",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)

In [ ]:
llama_trainer.cal_perplexity()

In [ ]:
path = Path("./experiment/llama/model").resolve()
llama_trainer = LanguageModelTrainer.load_model(
    device=get_device(),
    path=str(path),
)

In [ ]:
gen_config = GenerationConfig(
                temperature=0.7,
                top_p=0.9,
                top_k=30,
                # num_beams=5,
                max_new_tokens=256,
                repetition_penalty=1.1,
                do_sample=True,
                forced_decoder_token_ids=[
                    [llama_trainer.tokenizer.convert_tokens_to_ids("<")],  # Force opening bracket
                ],
            )

prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate(prompt=prompt,
                    model=llama_trainer.model,
                    gen_config=gen_config,
                    tokenizer=llama_trainer.tokenizer)
print(response)

In [ ]:
extract_equations_from_text(response)

In [ ]:
prompt = prepare_prompt(4)

In [ ]:
prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate(prompt=prompt, model=llama_trainer.model, tokenizer=llama_trainer.tokenizer)
print(response)

### DeepSeek

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
    "target_modules": ["q_proj", "v_proj"],
}


model_name = "deepseek-ai/deepseek-coder-1.3b-base"
deepSeek_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

deepSeek_trainer.setup()
final_exp_train_data = deepSeek_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = deepSeek_trainer.tokenize_dataset(exp_test_data)

# Train
deepSeek_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir="./experiment/deepseek",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)

In [ ]:
deepSeek_trainer.cal_perplexity()

In [ ]:
path = Path("./experiment/deepseek/deepseek-ai/model").resolve()
deepSeek_trainer = LanguageModelTrainer.load_model(
    device=get_device(),
    path=str(path),
)

In [ ]:
generator = prepare_generator(deepSeek_trainer.model, deepSeek_trainer.tokenizer)
prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate(prompt, generator, deepSeek_trainer.tokenizer)
print(response)

### Gemma 2b

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
    "target_modules": ["q_proj", "v_proj"],
}


model_name = "google/gemma-2b-it"
gemma_2b_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

gemma_2b_trainer.setup()
final_exp_train_data = gemma_2b_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = gemma_2b_trainer.tokenize_dataset(exp_test_data)

# Train
gemma_2b_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir=f"./experiment/gemma2b",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)


In [ ]:
gemma_2b_trainer.cal_perplexity()

In [ ]:
gemma_2b_trainer.tokenizer.eos_token_id

In [ ]:
path = Path("./experiment/gemma2b/model").resolve()
gemma_2b_trainer = LanguageModelTrainer.load_model(
    device=get_device(),
    path=str(path),
)

In [ ]:
prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate(prompt=prompt, model=gemma_2b_trainer.model, tokenizer=gemma_2b_trainer.tokenizer)
print(response)

In [ ]:
prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate(prompt=prompt, model=gemma_2b_trainer.model, tokenizer=gemma_2b_trainer.tokenizer)
print(response)

### GPT-NeoX-1.3B

In [ ]:

num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
    "target_modules": ["q_proj", "v_proj"],
}


model_name = "EleutherAI/gpt-neo-1.3B"
gpt_neo_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

gpt_neo_trainer.setup()
final_exp_train_data = gpt_neo_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = gpt_neo_trainer.tokenize_dataset(exp_test_data)

# Train
gpt_neo_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir=f"./experiment/gpt_neo",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)


In [ ]:
gpt_neo_trainer.cal_perplexity()

In [ ]:
from pathlib import Path

path = Path("./experiment/gpt_neo/EleutherAI/gpt_neo/").resolve()


In [ ]:
test_reload_trainer = LanguageModelTrainer.load_model(
    device=get_device(),
    path=str(path),
)

In [ ]:
generator = pipeline(
    "text-generation",
    model=test_reload_trainer.model,
    tokenizer=test_reload_trainer.tokenizer,
    device=get_device()
)

gen_config = GenerationConfig(
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    num_beams=5,
    max_new_tokens=256,
    repetition_penalty=1.1,
    do_sample=True
)

# 3. Generate text with a prompt
def generate_response(prompt, tokenizer):
    # Format the prompt as your model expects
    formatted_prompt = f"<s>{prompt}\n"

    # Generate text
    outputs = generator(
        formatted_prompt,
        generation_config=gen_config,
        return_full_text=False,  # Don't include the prompt in output
        pad_token_id=tokenizer.eos_token_id
    )

    return outputs[0]['generated_text']

In [ ]:
prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate_response(prompt, test_reload_trainer.tokenizer)
print(response)

### Phi-2

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
    "target_modules": ["q_proj", "v_proj"],
}


model_name = "microsoft/phi-2"
phi2_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

phi2_trainer.setup()
final_exp_train_data = phi2_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = phi2_trainer.tokenize_dataset(exp_test_data)

# Train
phi2_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir=f"./experiment/phi2",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)


In [ ]:
phi2_trainer.cal_perplexity()

In [ ]:
path = Path("./experiment/phi2/model").resolve()
phi2_trainer = LanguageModelTrainer.load_model(
    device=get_device(),
    path=str(path),
)

In [ ]:
prompt = prepare_prompt(4)
# print("Prompt:")
print(prompt)
response = generate(prompt=prompt, model=phi2_trainer.model, tokenizer=phi2_trainer.tokenizer)
print(response)

In [ ]:
prompt = prepare_prompt(5)
response = generate_multiple(
    prompt=prompt,
    model=phi2_trainer.model,
    tokenizer=phi2_trainer.tokenizer,
    num_variants=10
)


In [ ]:
for i in range(len(response)):
    print(f"Question {i+1} \n {response[i]}" )

In [ ]:
extract_equations_from_text(response)

## Experiment 2 (tuning Parameters for text generation per model)

In [ ]:
def generate_multiple_year_question(model, tokenizer):
    result = []
    for i in range(1, 10):
        tmpStore = []
        for idx in range(10):
            generator = prepare_generator(model, tokenizer)
            prompt = prepare_prompt(i)
            response = generate(prompt=prompt, generator=generator, tokenizer=tokenizer)
            tmpStore.append(prompt + response)
        result.append(tmpStore)

    return result

### PHI2

In [ ]:
# Example usage

model_name = "microsoft/phi-2"
phi2_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name
)

phi2_trainer.setup()

# Initialize the optimizer
optimizer = HyperparameterOptimizer(
    trainer_class=LanguageModelTrainer,
    device=get_device(),
    model_name=model_name
)


final_exp_train_data = phi2_trainer.tokenize_dataset(exp_train_data)
final_exp_val_data = phi2_trainer.tokenize_dataset(exp_val_data)


# Run optimization
phi2_best_params = optimizer.optimize(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_val_data,
    output_dir_base="./phi2_optuna",
    n_trials=30,
)



In [ ]:
phi2_best_params

In [ ]:
!zip -r llama_optuna.zip llama_optuna

In [ ]:
# Example usage

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llama_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name
)

llama_trainer.setup()
final_exp_train_data = llama_trainer.tokenize_dataset(exp_train_data)
final_exp_val_data = llama_trainer.tokenize_dataset(exp_val_data)


# Initialize the optimizer
llamal_optimizer = HyperparameterOptimizer(
    trainer_class=LanguageModelTrainer,
    device=get_device(),
    model_name=model_name
)

# Run optimization
llama_best_params = llamal_optimizer.optimize(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_val_data,
    output_dir_base="./llama_optuna",
    n_trials=30,
)



In [ ]:
llama_best_params

In [ ]:
result = generate_multiple_year_question(phi2_trainer.model, phi2_trainer.tokenizer)

In [ ]:

generator = prepare_generator(phi2_trainer.model, phi2_trainer.tokenizer)
prompt = prepare_prompt(4)
print("Prompt:")
print(prompt)
response = generate(prompt=prompt, generator=generator, tokenizer=phi2_trainer.tokenizer)
print(response)

In [ ]:
import optuna
study = optuna.load_study(study_name="your_study_name", storage="/llama_optuna/study.db")
print("Best trial:", study.best_trial.params)

## Experiment 3 with Selected model

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
    "target_modules": ["fc1", "fc2"],
}


model_name = "microsoft/phi-2"
phi2_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

phi2_trainer.setup()
final_exp_train_data = phi2_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = phi2_trainer.tokenize_dataset(exp_test_data)

# Train
phi2_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir=f"./experiment_3/phi2/t1",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)


In [ ]:
phi2_trainer.cal_perplexity()

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
        "r": 32,
        "lora_alpha": 8,
        "lora_dropout": 0.034,
        "bias": "none",
        "target_modules": ["q_proj", "v_proj"]
}


model_name = "microsoft/phi-2"
phi2_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

phi2_trainer.setup()
final_exp_train_data = phi2_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = phi2_trainer.tokenize_dataset(exp_test_data)

# Train
phi2_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir=f"./experiment_3/phi2/t2",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)


In [ ]:
phi2_trainer.cal_perplexity()

In [ ]:
num_train_samples = len(exp_train_data)
per_device_batch_size = 4
num_devices = 1  # adjust for multi-GPU
gradient_accumulation_steps = 1
epochs = 3

# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Dynamic evaluation every half epoch
eval_steps = steps_per_epoch // 2


lora_params = {
        "r": 32,
        "lora_alpha": 8,
        "lora_dropout": 0.034,
        "bias": "none",
        "target_modules": ["q_proj", "v_proj","fc1", "fc2"]
}


model_name = "microsoft/phi-2"
phi2_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name=model_name,
    tokenizer_name=model_name,
    lora_config=lora_params
)

phi2_trainer.setup()
final_exp_train_data = phi2_trainer.tokenize_dataset(exp_train_data)
final_exp_test_data = phi2_trainer.tokenize_dataset(exp_test_data)

# Train
phi2_trainer.train(
    train_dataset=final_exp_train_data,
    eval_dataset=final_exp_test_data,
    output_dir=f"./experiment_3/phi2/t2",
    training_args={
        "per_device_train_batch_size": per_device_batch_size,
        "num_train_epochs": 3,
        "eval_steps": eval_steps,
        "logging_steps": int(steps_per_epoch/epochs)
    }
)


In [ ]:
phi2_trainer.cal_perplexity()

## Training Model with full dataset


In [ ]:
num_train_samples = len(train_prompts)
per_device_batch_size = 4
num_devices=1
gradient_accumulation_steps = 1
epochs = 3
# Compute total steps per epoch
steps_per_epoch = (num_train_samples // (per_device_batch_size * num_devices * gradient_accumulation_steps))

# Auto-detect devices
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    per_device_batch_size = 4  # Keep this small for Colab's GPU memory
    # Calculate effective batch size
    effective_batch_size = per_device_batch_size * num_devices * gradient_accumulation_steps
    steps_per_epoch = (num_train_samples // effective_batch_size)



# Initialize trainer with best parameters
final_trainer = LanguageModelTrainer(
    device=get_device(),
    model_name="microsoft/phi-2",
    tokenizer_name=None,  # Will use phi-2's tokenizer
    lora_config={
        "r": 32,
        "lora_alpha": 8,
        "lora_dropout": 0.034,
        "bias": "none",
        "target_modules": ["q_proj", "v_proj","fc1", "fc2"]
    }
)



# Setup the model
final_trainer.setup(trust_remote_code=True)  # Required for Phi-2

train_dataset = Dataset.from_list(train_prompts)
test_dataset = Dataset.from_list(train_prompts)

# Tokenize datasets
train_dataset = final_trainer.tokenize_dataset(train_dataset)
eval_dataset = final_trainer.tokenize_dataset(test_dataset)

# Train with optimized parameters
final_trainer.train(
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    output_dir="./phi2_final_model",
    training_args={
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 4,
        "learning_rate": 2.28e-4,
        "num_train_epochs": 3,
        "gradient_accumulation_steps": 1,
        "warmup_steps": 438,
        "weight_decay": 0.0638,
        "optim": "adafactor",
        "fp16": True,
        "logging_steps": 50,
        "eval_steps": steps_per_epoch//10,
        "save_strategy": "epoch",
        "output_dir": "./phi2_final_model",
        "report_to": "tensorboard"
    }
)


In [ ]:
final_trainer.cal_perplexity()

In [ ]:
# Load the trained model
final_trainer = LanguageModelTrainer.load_model(
    path="./model",
    device=get_device()
)

# # Calculate final perplexity
# final_perplexity = final_trainer.cal_perplexity()
# print(f"Final Model Perplexity: {final_perplexity:.2f}")

# Generate year 1 - 10 Mertial

In [ ]:
 # define parameters for response
generation_context = {}

for i in tqdm(range(1, 11), desc="Generating questions for each year"):
    print(f"Generating questions for year {i}")
    prompt = prepare_prompt(i)
    print("Prompt:")
    print(prompt)


    response = generate_multiple(
        prompt=prompt,
        model=final_trainer.model,
        tokenizer=final_trainer.tokenizer,
        num_variants=10
    )

    generation_context[i] = response

# Evulation

In [ ]:
from PHI2_Utils import PHI2_Utils
import json
import numpy as np

phi2_utils = PHI2_Utils()


with open("desc.json", "r") as f:
    desc_data = json.load(f)

In [ ]:
embedded_desc = []
for i in range(len(desc_data)):
    print(desc_data[i]['year'], desc_data[i]['scope'])
    scope = desc_data[i]['scope']
    result = {
        "year": desc_data[i]['year'],
        "scope": phi2_utils.embed(scope)
    }
    embedded_desc.append(result)

In [ ]:
# loop each question in generation_context to embed
embedded_generation_context = []
for year, questions in generation_context.items():
    for question in questions:
        result = {
            "year": year,
            "question": phi2_utils.embed(question)
        }
        embedded_generation_context.append(result)

In [ ]:
# export the embedded_desc and embedded_generation_context to json


# Convert numpy arrays to lists before saving
embedded_desc_list = []
for item in embedded_desc:
    embedded_desc_list.append({
        "year": item["year"],
        "scope": item["scope"].tolist() # Convert numpy array to list
    })

embedded_generation_context_list = []
for item in embedded_generation_context:
    embedded_generation_context_list.append({
        "year": item["year"],
        "question": item["question"].tolist() # Convert numpy array to list
    })


with open("embedded_desc.json", "w") as f:
    json.dump(embedded_desc_list, f)

with open("embedded_generation_context.json", "w") as f:
    json.dump(embedded_generation_context_list, f)

In [ ]:
# loaed the embedded_desc and embedded_generation_context from json
with open("curriculum_Desc/embedded_desc.json", "r") as f:
    embedded_desc = json.load(f)
with open("curriculum_Desc/embedded_generation_context.json", "r") as f:
    embedded_generation_context = json.load(f)

In [ ]:
for question in embedded_generation_context:
    print(question['year'])

In [ ]:
# calculate cosine similarity with phi2_utils.cosine_similarity
similarity_results = []
for desc in embedded_desc:
    question_similarities = []
    for question in embedded_generation_context:
        desc_year = desc['year'].split()[-1]  # Extract number from "year X"
        question_year = str(question['year']).split(':')[-1]  # Extract number from "year:X"
        if desc_year == question_year:
            # print(desc['year'])
            similarity = phi2_utils.cosine_similarity(np.array(desc['scope']), np.array(question['question']))
            result = {
                "year": desc['year'],
                "similarity": similarity,
                "question": question['question']
            }
            question_similarities.append(result)
    similarity_results.append(question_similarities)


In [ ]:
for items in similarity_results:
   for item in items:
      if item['similarity'] > 0.7:
         print(item)

# Generate Question that over 0.7 Similarity

In [ ]:
final_trainer = LanguageModelTrainer.load_model(
    path="./model",
    device=get_device()
)

In [ ]:
 # define parameters for response
generation_context = {}


generate_random_student_levels()
for i in tqdm(range(1, 11), desc="Generating questions for each year"):
    print(f"Generating questions for year {i}")
    prompt = prepare_prompt(i)
    print("Prompt:")
    print(prompt)


    response = generate_multiple(
        prompt=prompt,
        model=final_trainer.model,
        tokenizer=final_trainer.tokenizer,
        num_variants=10
    )

    generation_context[i] = response

In [ ]:
# export the generation_context
with open("generation_context_v2.json", "w") as f:
    json.dump(generation_context, f)

In [ ]:
from PHI2_Utils import PHI2_Utils
import json
import numpy as np

phi2_utils = PHI2_Utils()


with open("embedded_desc.json", "r") as f:
    embedded_desc = json.load(f)

In [ ]:
embedded_ques = {}
for year, questions in generation_context.items():
    embedded_questions_list = []
    for question in questions:
        result = {
            "year": year,
            "question": phi2_utils.embed(question)
        }
        embedded_questions_list.append(result)
    embedded_ques[year] = embedded_questions_list

In [ ]:
embedded_ques

In [ ]:
# export the embedded_ques

# Convert numpy arrays to lists before saving
embedded_ques_list = {}
for year, questions_list in embedded_ques.items():
    embedded_questions_list_of_lists = []
    for question_data in questions_list:
        embedded_questions_list_of_lists.append({
            "year": question_data["year"],
            "question": question_data["question"].tolist()  # Convert numpy array to list
        })
    embedded_ques_list[year] = embedded_questions_list_of_lists




In [ ]:
with open("embedded_ques_v2.json", "w") as f:
    json.dump(embedded_ques_list, f)

In [ ]:
# use phi2_utils to calculate the similarity
# calculate cosine similarity with phi2_utils.cosine_similarity
similarity_results = []
for desc in embedded_desc:
    desc_year = desc['year'].split()[-1]  # Extract number from "year X"
    question_similarities = []
    if int(desc_year) in embedded_ques_list:
        for question_data in embedded_ques_list[int(desc_year)]:
            question_year = str(question_data['year'])  # Get the year from the question data
            # Only compare questions from the same year
            if desc_year == question_year:
                similarity = phi2_utils.cosine_similarity(np.array(desc['scope']), np.array(question_data['question']))
                result = {
                    "year": desc['year'],
                    "similarity": similarity,
                    "question": question_data['question']
                }
                question_similarities.append(result)
    similarity_results.append(question_similarities)

In [ ]:
for items in similarity_results:
   for item in items:
      if item['similarity'] < 0.7:
         print(item)

# Generation V3

In [ ]:
final_trainer = LanguageModelTrainer.load_model(
    path="./model",
    device=get_device()
)

In [ ]:
# initial attempt record
attempt_cnt = {
    "year1":0,
    "year2":0,
    "year3":0,
    "year4":0,
    "year5":0,
    "year6":0,
    "year7":0,
    "year8":0,
    "year9":0,
    "year10":0
    }

In [ ]:
# initial questions list for storing all generation context
questions_list = {
    "year1":[],
    "year2":[],
    "year3":[],
    "year4":[],
    "year5":[],
    "year6":[],
    "year7":[],
    "year8":[],
    "year9":[],
    "year10":[]
}

## Generating questions for year 1

In [ ]:
 # define parameters for response
target_year = 1
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year1'].append(response[1])

## Generating questions for year 2

In [ ]:
 # define parameters for response
target_year = 2
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year2'].append(response[2])

## Generating questions for year 3

In [ ]:
 # define parameters for response
target_year = 3
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year3'].append(response[3])

## Generating questions for year 4

In [ ]:
 # define parameters for response
target_year = 4
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year4'].append(response[4])

## Generating questions for year 5

In [ ]:
 # define parameters for response
target_year = 5
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list["year5"].append(response[5])

In [ ]:
questions_list["year5"]

## Generating questions for year 6

In [ ]:
 # define parameters for response
target_year = 6
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
len(response[6])

In [ ]:
questions_list['year6'].append(response[6])

## Generating questions for year 7

In [ ]:
 # define parameters for response
target_year = 7
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year7'].append(response[7])

## Generating questions for year 8

In [ ]:
 # define parameters for response
target_year = 8
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year8'].append(response[8])

## Generating questions for year 9

In [ ]:
 # define parameters for response
target_year = 9
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year9'].append(response[9])

## Generating questions for year 10

In [ ]:
 # define parameters for response
target_year = 10
attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)


In [ ]:
questions_list['year10'].append(response[10])

## Generation Result

In [ ]:
# export the question_list
with open("questions_list.json", "w") as f:
    json.dump(questions_list, f)

In [ ]:
import json
# load the questions_list.json
with open("./curriculum_Desc/questions_list.json", "r") as f:
    questions_list = json.load(f)

In [ ]:
for i in range(1, 11):
    print(f"Year {i}  No. of Questions {len(questions_list[f'year{i}'][0])}"   )

The ratio of generate content that over 0.7 similarity with the Curriculum Standard
* Year 1 : 10 in 47
* Year 2 : 10 in 150
* Year 3 : 5 in 400
* Year 4 : 1 in 400
* Year 5 : 10 in 102
* Year 6 : 10 in 218
* Year 7 : 10 in 18
* Year 8 : 10 in 50
* Year 9 : 10 in 58
* Year 10 : 10 in 100

### Generation data for final fine-tuning

In [ ]:
questions_list = []

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=100)
questions_list.append(response[10])

In [ ]:
len(questions_list)

In [ ]:
# export questions_list
with open("generation_v3_data.json", "w") as f:
    json.dump(questions_list, f)

In [ ]:
embedded_ques = {}
for year, questions in generation_context.items():
    embedded_questions_list = []
    for question in questions:
        result = {
            "year": year,
            "question": phi2_utils.embed(question)
        }
        embedded_questions_list.append(result)
    embedded_ques[year] = embedded_questions_list

In [ ]:
embedded_ques_list = {}
for year, questions_list in embedded_ques.items():
    embedded_questions_list_of_lists = []
    for question_data in questions_list:
        embedded_questions_list_of_lists.append({
            "year": question_data["year"],
            "question": question_data["question"].tolist()  # Convert numpy array to list
        })
    embedded_ques_list[year] = embedded_questions_list_of_lists

In [ ]:
# use phi2_utils to calculate the similarity
# calculate cosine similarity with phi2_utils.cosine_similarity
similarity_results = []
for desc in embedded_desc:
    desc_year = desc['year'].split()[-1]  # Extract number from "year X"
    question_similarities = []
    if int(desc_year) in embedded_ques_list:
        for question_data in embedded_ques_list[int(desc_year)]:
            question_year = str(question_data['year'])  # Get the year from the question data
            # Only compare questions from the same year
            if desc_year == question_year:
                similarity = phi2_utils.cosine_similarity(np.array(desc['scope']), np.array(question_data['question']))
                result = {
                    "year": desc['year'],
                    "similarity": similarity,
                    "question": question_data['question']
                }
                question_similarities.append(result)
    similarity_results.append(question_similarities)

In [ ]:
for items in similarity_results:
   for item in items:
      if item['similarity'] > 0.7:
         print(item)

# Final Fine-Tuning

In [ ]:
import json
# read the generation_v3_data.json
with open("generation_v3_data.json", "r") as f:
    generation_context = json.load(f)

In [ ]:
# loop the generation_context to signal array
data = [item for sublist in generation_context for item in sublist]

In [ ]:
# export data
# export questions_list
with open("all_data.json", "w") as f:
    json.dump(data, f)


In [ ]:
data[0]

In [ ]:
# from feedback_fine_tuning import feedback_fine_tune_phi2
# reload feedback_fine_tune_phi2
import sys
import importlib
importlib.reload(sys.modules["feedback_fine_tuning"])
from feedback_fine_tuning import feedback_fine_tune_phi2

In [ ]:
feedback_fine_tune_phi2(
    base_model_path="model",  # Use your fine-tuned model
    output_dir="./phi2_feedback_v2_model",
    similarity_threshold=0.7,
    training_method="supervised",
    existing_data_path="all_data.json",
    default_year=10
)


In [ ]:
!zip -r phi2_feedback_v2_model.zip phi2_feedback_v2_model

In [ ]:
from google.colab import files
files.download('phi2_feedback_v2_model.zip')

### evluation

In [ ]:
final_trainer = LanguageModelTrainer.load_model(
    path="./phi2_feedback_v2_model/model",
    device=get_device()
)

In [ ]:
 # define parameters for response
target_year = 10
# attempt_cnt[f"year{target_year}"] = attempt_cnt[f"year{target_year}"] + 1
response = generate_questions_for_specific_years(model=final_trainer.model,tokenizer=final_trainer.tokenizer, target_year=target_year, max_total_generations=10)

In [ ]:
response

In [ ]:
response